In [ ]:
from tkinter import Tk
from tkinter.filedialog import askopenfilename
from dateutil.relativedelta import relativedelta
import calendar
import pandas as pd
import numpy as np

# Loading all the data on the system
class PD_Modelling:
    def __init__(self):
        self.loan_listing = None
        self.monthly = None
        self.sorted_data = None
        self.Transition_Total = None
        self.PD_per_month = None
        self.Cure_rate = None
        self.PD_Normalized = None
        self.PD_powered = None
        self.Sorted_Matrix = None
        self.Matrix = None

    def load_loan_data(self):
        Tk().withdraw()
        print("Please select the loan listing data") 
        file = askopenfilename()
        all_sheets = pd.read_excel(file,sheet_name=None)
        self.loan_listing = pd.concat(all_sheets.values(), ignore_index=True)

    def sorting_values(self):
        self.loan_listing['As_At'] = pd.to_datetime(self.loan_listing['As_At'])
        self.sorted_data = self.loan_listing.groupby('ACCOUNT NO').apply(lambda x: x.sort_values('As_At'), include_groups = True).reset_index(drop=True)
        #return self.sorted_data
    
    def create_new_month_staging(self):
        self.sorted_data['New_Month_Staging'] = np.where(
        self.sorted_data['ACCOUNT NO'] == self.sorted_data['ACCOUNT NO'].shift(-1),
        self.sorted_data['Staging'].shift(-1),
            'Offbooks')
        #return self.sorted_data

    def monthly_transitions(self):
        self.monthly = pd.crosstab(index=[self.sorted_data.SEGMENT,self.sorted_data.As_At, self.sorted_data.Staging], 
                                       columns=[self.sorted_data.New_Month_Staging],
                                       values=self.sorted_data.BALANCE, 
                                       aggfunc='sum')
        self.monthly.reset_index(inplace=True)
        self.monthly.fillna(0, inplace=True)
        self.monthly.rename(columns= {'Staging':'Transition'}, inplace=True)

    def pivot_data(self):
        self.Transition_Total = pd.crosstab(index=[self.sorted_data.SEGMENT, self.sorted_data.Staging], 
                                       columns=[self.sorted_data.New_Month_Staging],
                                       values=self.sorted_data.BALANCE, 
                                       aggfunc='sum')
        self.Transition_Total.reset_index(inplace=True)
        self.Transition_Total.fillna(0, inplace=True)  # Replacing NaN values by 0s
        self.Transition_Total.drop(columns=['Offbooks'], inplace=True)  # Dropping the offbook column
        
    def process_data(self):
        # Step 2: Obtaining row percentages for every transition, and normalizing the matrix
        self.PD_per_month = self.Transition_Total.loc[:,:]
        self.PD_per_month[['Stage 1','Stage 2','Stage 3']] = \
            self.Transition_Total[['Stage 1', 'Stage 2', 'Stage 3']].apply(lambda x: x / x.sum(), axis=1)
        
        
        # Step 3: Storing cure rates per sector to be used for LGD
        self.Cure_rate = self.PD_per_month.loc[:,:]
        self.Cure_rate['Cure rate'] = self.PD_per_month.apply(
            lambda row: row['Stage 1'] + row['Stage 2'] if row['Staging'] == 'Stage 3' else None, axis=1
        )
        self.Cure_rate = self.Cure_rate[['SEGMENT', 'Cure rate']]
        #self.Transition_Total.drop(columns=['Cure rate'], inplace=True)  # Dropping the Cure rate column
        self.Cure_rate = self.Cure_rate.dropna()

        # Step 4: Normalizing matrices
        self.PD_Normalized = pd.DataFrame(self.PD_per_month.loc[:,:]) 
         
        def set_value(row):
            if row['Staging'] == 'Stage 3':
                for col in ['Stage 1','Stage 2']:
                    row[col] = 0

            return row
        self.PD_Normalized = self.PD_Normalized.apply(set_value, axis=1)

        self.PD_Normalized.loc[self.PD_Normalized['Staging'] == 'Stage 3','Stage 3'] = 1 # Assigning 100% Default Rate to the loans from Stage 3



        #mask = self.PD_Normalized['Staging'] == 'Stage 3'
        #self.PD_Normalized = self.PD_Normalized.loc[mask, ['Stage 3', 'Stage 1', 'Stage 2']] = [1, 0, 0]
        
        #self.PD_per_month.loc[self.PD_per_month['Staging'].isin(['Stage 1']), 'Default'] = 0
        #self.PD_per_month.loc[self.PD_per_month['IFRS_9_Staging'] == '0-29 dpd', '60-89 dpd'] = 0

        
    def calculate_matrices(self):
        # Step 5: Multiplying matrices by each other according to the Markov property
        sectors = self.PD_Normalized['SEGMENT'].unique()  # Extract unique sectors
        matrices = {}  # Create dictionary to store matrices for each sector

        # Loop through sectors and create matrices
        for sector in sectors:
            sector_data = self.PD_Normalized[self.PD_Normalized['SEGMENT'] == sector].iloc[:, 2:].to_numpy()  # Convert to numpy array
            matrices[sector] = np.concatenate(
                [sector_data] + [np.linalg.matrix_power(sector_data, i + 2) for i in range(299)], axis=0
            )
        
        PD_powered = []  # Create a list to hold the DataFrames for each sector

        for sector in sectors:
            sector_matrix = matrices[sector]
            sector_df = pd.DataFrame(sector_matrix, index=[sector] * sector_matrix.shape[0])
            PD_powered.append(sector_df)

        self.PD_powered = pd.concat(PD_powered)
        self.Matrix = self.PD_powered.reset_index()
        self.Matrix.rename(columns={'index': 'SECTOR', 0: 'Stage 1', 1: 'Stage 2', 2: 'Stage 3'}, inplace=True)
        self.Matrix.insert(1, 'Month', self.Matrix.groupby('SECTOR').cumcount() // 3 + 1)  # Cumulatively count the matrix

        # Creating a list of values to repeat
        values_to_repeat = ['Stage 1', 'Stage 2', 'Stage 3']
        length = len(self.Matrix)
        repeated_values = [values_to_repeat[i % len(values_to_repeat)] for i in range(length)]
        self.Matrix.insert(2, 'Transition', repeated_values)  # Assign a new column called Transition

        # Sorting the matrix
        self.Sorted_Matrix = self.Matrix.sort_values(['SECTOR', 'Transition'], ascending=[True, True])
        self.Sorted_Matrix.reset_index(drop=True, inplace=True)
        self.Sorted_Matrix['Marginal_PD'] = np.nan
        for i in range(len(self.Sorted_Matrix)):
            if i == 0:
                #For the first row, Month is 1
                self.Sorted_Matrix.at[i, 'Marginal_PD'] = self.Sorted_Matrix.at[i, 'Stage 3']
            else:
                # For subsequent rows, use the previous row's 'Default' value
                current_sector = self.Sorted_Matrix.at[i, 'SECTOR']
                current_stage = self.Sorted_Matrix.at[i, 'Transition']
                prev_sector = self.Sorted_Matrix.at[i - 1, 'SECTOR']
                prev_stage = self.Sorted_Matrix.at[i - 1, 'Transition']
                 
                if current_sector != prev_sector or current_stage != prev_stage:
                  # If the sector or stage has changed, use the 'Default' value directly
                    self.Sorted_Matrix.at[i, 'Marginal_PD'] = self.Sorted_Matrix.at[i, 'Stage 3']
                else:
                    # For subsequent rows within the same sector and stage, use the previous row's 'Default' value
                    prev_default = self.Sorted_Matrix.at[i - 1, 'Stage 3']
                    if prev_default == 1:
                # Avoid division by zero
                        self.Sorted_Matrix.at[i, 'Marginal_PD'] = 1
                    else:
                        self.Sorted_Matrix.at[i, 'Marginal_PD'] = (self.Sorted_Matrix.at[i, 'Stage 3'] - prev_default) / (1 - prev_default)
                        if (1 - prev_default) == 0:
                            self.Sorted_Matrix.at[i, 'Marginal_PD'] = 0


    def get_results(self):
        # Return the results for external access
        return {
            'Sorted_Data':self.sorted_data,
            'Monthly Transition':self.monthly,
            'Summary_Transition':self.Transition_Total,
            'PD_per_month': self.PD_per_month,
            'Cure_rate': self.Cure_rate,
            'PD_Normalized': self.PD_Normalized,
            'PD_powered': self.PD_powered,
            'Matrix': self.Matrix,
            'Sorted_Matrix': self.Sorted_Matrix
        }

# Usage example
modeling = PD_Modelling()
modeling.load_loan_data()
modeling.sorting_values()
modeling.create_new_month_staging()
modeling.monthly_transitions()
modeling.pivot_data()
modeling.process_data()
modeling.calculate_matrices()

results = modeling.get_results()



#Accessing data frames outside the class
SortedData = results['Sorted_Data']

SummaryTransition = results['Summary_Transition']
monthly_transition = results['Monthly Transition']
pd_per_month = results['PD_per_month']
cure_rate = results['Cure_rate']
pd_normalized = results['PD_Normalized']
pd_powered = results['PD_powered']
matrix = results['Matrix']
sorted_matrix = results['Sorted_Matrix']


  
    
#loader = LoanDataLoader()
#loader.load_loan_data()
#loader.sorting_values()
#]combined_data = loader.create_new_month_staging()

Please select the loan listing data


C:\Users\agitaka001\AppData\Local\Temp\ipykernel_26448\3433567257.py:31: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  self.sorted_data = self.loan_listing.groupby('ACCOUNT NO').apply(lambda x: x.sort_values('As_At'), include_groups = True).reset_index(drop=True)


In [ ]:
display(sorted_matrix)

""


In [ ]:
from tkinter import Tk
from tkinter.filedialog import askopenfilename
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np

class ECL_Modelling:
    def __init__(self):
        self.df = None

    def loading(self):
        Tk().withdraw()
        print("Please select the historical data to load") 
        file = askopenfilename()
        self.df = pd.read_excel(file, header=0)

    def generate_snapshot_dates(self, row):
        snapshot_dates = []
        current_date = row['Reporting_date']
        maturity_date = row['Adjusted_maturity_date']
        stage = row['Stage']

        if stage == 'Stage 1':
            for _ in range(13):
                snapshot_dates.append(current_date)
                current_date += relativedelta(months=1, day=31)
                if current_date > maturity_date:
                    break
        elif stage in ['Stage 2']:
            while current_date <= maturity_date:
                snapshot_dates.append(current_date)
                current_date += relativedelta(months=1, day=31)
        
        return pd.Series(snapshot_dates)

    def calculate_period_since_origination(self, row):
        return ((row['Snapshot Dates'].year - row['First_payment'].year) * 12 +
                (row['Snapshot Dates'].month - row['First_payment'].month)) + 1

    def calculate_monthly_installment(self, row):
        n_months = ((row['Reporting_date'].year - row['First_payment'].year) * 12 + 
                     (row['Reporting_date'].month - row['First_payment'].month))

        term_to_maturity = ((row['Adjusted_maturity_date'].year - row['First_payment'].year) * 12 + 
                            (row['Adjusted_maturity_date'].month - row['First_payment'].month))
        
        months_difference = term_to_maturity - n_months

        if months_difference <= 0:
            return 0

        monthly_rate = row['Effective_IR'] / 12
        if monthly_rate > 0:
            installment = (row['Outstanding_Amount'] * monthly_rate) / (1 - (1 + monthly_rate) ** -months_difference)
        else:
            installment = row['Outstanding_Amount'] / months_difference
        
        return installment
    
    def calculate_balance(self, row, previous_balance, previous_account_no, monthly_installment):
        if row['TRANS_ID'] != previous_account_no:
            return row['Outstanding_Amount']
        else:
            return previous_balance * (1 + row['Effective_IR'] / 12) - monthly_installment
    
    def calculate_balance_after_missed_payment(self,EIRs, account_numbers, stages):
        balances_after_missed_payment = []
        for trans_id in np.unique(account_numbers):
            trans_df = self.df[self.df['TRANS_ID'] == trans_id]
            for i in range(len(trans_df)):
                offset_index = i - min(3, trans_df.loc[i, 'Period since origination'] - trans_df.loc[0, 'Period since origination'])
                if offset_index >= 0:
                    balance = trans_df.loc[offset_index, 'Balance after repayment'] * (1 + EIRs[i]) ** min(3, trans_df.loc[i, 'Period since origination'] - trans_df.loc[0, 'Period since origination'])
                else:
                    balance = trans_df.loc[i, 'Balance after repayment']
                balances_after_missed_payment.append(balance)
        return balances_after_missed_payment

class MergedDataframeHandler:
    def __init__(self, merged_df):
        self.merged_df = merged_df
        Tk().withdraw()
        print("Please enter the marginal PDs")
        file = askopenfilename()
        self.pd_marginal = pd.read_excel(file, header = 0)

    def merge_dataframes(self, final_df):
        merged_df = final_df.merge(self.
            pd_marginal[['SECTOR', 'Month', 'Marginal_PD', 'Transition']],
            left_on=['Class', 'Period since origination', 'Stage'],
            right_on=['SECTOR', 'Month', 'Transition'],
            how='left'
        )
        merged_df.drop(columns=['SECTOR', 'Month', 'Transition'], inplace=True, errors='ignore')
        return merged_df

    def perform_new_calculations(self):
        self.merged_df['LGW'] = np.where(self.merged_df['Balance after missed payment'] != 0,
                                          np.maximum(0, (1 - (self.merged_df['Total DFSV'] / self.merged_df['Balance after missed payment']))), 0)
        self.merged_df['LGD'] = np.maximum(0, self.merged_df['LGW'] * (1 - self.merged_df['Cure rate']))
        self.merged_df['Credit Loss'] = self.merged_df['Balance after missed payment'] * self.merged_df['Marginal_PD'] * self.merged_df['LGD']
        self.merged_df['Discounted ECL'] = self.merged_df.apply(self.calculate_discounted_ecl, axis=1)

    def create_period_to_be_discounted(self):
        self.merged_df['Period to be discounted'] = self.merged_df.groupby('ACCOUNT NO').cumcount()

    def calculate_discounted_ecl(self, row):
        if row['Stage'] == 'Stage 3':
            return row['Credit Loss']
        if row['Stage'] in ['Stage 1', 'Stage 2']:
            eir = row['EIR']
            period_to_discount = row['Period to be discounted']
            if period_to_discount > 0:
                return row['Credit Loss'] * (1 + eir / 12) ** -period_to_discount
        return 0

# Main Execution
ecl_model = ECL_Modelling()
ecl_model.loading()

# Preallocate arrays
account_numbers, snapshot_dates, first_payments, maturity_dates = [], [], [], []
EIRs, stages, classes, cure_rates, dfsv, balances = [], [], [], [], [], []
periods_since_origination, monthly_installments, adjusted_balances = [], [], []

previous_account_no = None
previous_balance = 0

for index, row in ecl_model.df.iterrows():
    snapshots = ecl_model.generate_snapshot_dates(row)
    length = len(snapshots)

    account_numbers.extend([row['TRANS_ID']] * length)
    snapshot_dates.extend(snapshots)
    first_payments.extend([row['First_payment']] * length)
    maturity_dates.extend([row['Adjusted_maturity_date']] * length)
    EIRs.extend([row['Effective_IR']] * length)
    stages.extend([row['Stage']] * length)
    classes.extend([row['Sector']] * length)
    cure_rates.extend([row['Cure_rate']] * length)
    dfsv.extend([row['Total_DFSV']] * length)
    balances.extend([row['Outstanding_Amount']] * length)

    monthly_installment = ecl_model.calculate_monthly_installment(row)

    for snapshot_date in snapshots:
        period = ecl_model.calculate_period_since_origination({
            'Snapshot Dates': snapshot_date,
            'First_payment': row['First_payment']
        })
        periods_since_origination.append(period)
        monthly_installments.append(monthly_installment)
        adjusted_balance = ecl_model.calculate_balance(row, previous_balance, previous_account_no, monthly_installment)
        adjusted_balances.append(adjusted_balance)
        previous_balance = adjusted_balance
        previous_account_no = row['TRANS_ID']

# Create final DataFrame
                
final_df = pd.DataFrame({
    'ACCOUNT NO': account_numbers,
    'Snapshot Dates': snapshot_dates,
    'First_payment': first_payments,
    'Maturity date': maturity_dates,
    'EIR': EIRs,
    'Stage': stages,
    'Class': classes,
    'Cure rate': cure_rates,
    'Total DFSV': dfsv,
    'Original Outstanding Balance': balances,
    'Period since origination': periods_since_origination,
    'Monthly installment': monthly_installments,
    'Balance after repayment': adjusted_balances,
    'Balance after missed payment':balances_after_missed_payment
})

balances_after_missed_payment = ecl_model.calculate_balance_after_missed_payment(EIRs, account_numbers, stages)
final_df['Balance after missed payment'] = balances_after_missed_payment

# Merge with sorted_matrix
#sorted_matrix = pd.DataFrame()  # Ensure sorted_matrix is defined
merged_handler = MergedDataframeHandler(final_df)
merged_df = merged_handler.merge_dataframes(final_df)

# Handle the merged dataframe
merged_handler.merged_df = merged_df
merged_handler.create_period_to_be_discounted()
merged_handler.perform_new_calculations()

Please select the historical data to load


KeyError: 'Period since origination'

In [ ]:
print("Columns in final_df:", final_df.columns)


Columns in final_df: Index(['ACCOUNT NO', 'Snapshot Dates', 'First_payment', 'Maturity date', 'EIR',
       'Stage', 'Class', 'Cure rate', 'Total DFSV',
       'Original Outstanding Balance', 'Period since origination',
       'Monthly installment', 'Balance after repayment',
       'Balance after missed payment'],
      dtype='object')


In [ ]:
display(merged_df)

,ACCOUNT NO,Snapshot Dates,First_payment,Maturity date,EIR,Stage,Class,Cure rate,Total DFSV,Original Outstanding Balance,Period since origination,Monthly installment,Balance after repayment,Balance after missed payment,Marginal_PD,Period to be discounted,LGW,LGD,Credit Loss,Discounted ECL
0,AA23020SWMKM,2024-09-30,2021-07-31,2031-06-30,0.150,Stage 1,CONSTRUCTION,0.006404,4.111658e+10,4.350960e+10,39,8.572940e+08,4.350960e+10,4.350960e+10,0.001173,0,0.055,0.054648,2.788435e+06,0.000000e+00
1,AA23020SWMKM,2024-10-31,2021-07-31,2031-06-30,0.150,Stage 1,CONSTRUCTION,0.006404,4.111658e+10,4.350960e+10,40,8.572940e+08,4.319618e+10,4.350960e+10,0.001173,1,0.055,0.054648,2.788435e+06,2.754010e+06
2,AA23020SWMKM,2024-11-30,2021-07-31,2031-06-30,0.150,Stage 1,CONSTRUCTION,0.006404,4.111658e+10,4.350960e+10,41,8.572940e+08,4.287884e+10,4.350960e+10,0.001173,2,0.055,0.054648,2.788435e+06,2.720010e+06
3,AA23020SWMKM,2024-12-31,2021-07-31,2031-06-30,0.150,Stage 1,CONSTRUCTION,0.006404,4.111658e+10,4.350960e+10,42,8.572940e+08,4.255753e+10,4.350960e+10,0.001173,3,0.055,0.054648,2.788435e+06,2.686429e+06
4,AA23020SWMKM,2025-01-31,2021-07-31,2031-06-30,0.150,Stage 1,CONSTRUCTION,0.006404,4.111658e+10,4.350960e+10,43,8.572940e+08,4.223220e+10,4.350960e+10,0.001173,4,0.055,0.054648,2.788435e+06,2.653263e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
340888,AA23048G2D01,2024-09-30,2023-03-31,2024-08-31,0.175,Stage 1,CONSUMER,0.054073,0.000000e+00,1.163000e+03,19,0.000000e+00,1.163000e+03,1.163000e+03,0.004574,0,1.000,0.945927,5.031886e+00,0.000000e+00
340889,AA230202X957,2024-09-30,2016-04-30,2023-01-31,0.180,Stage 1,CONSUMER,0.054073,0.000000e+00,1.085000e+03,102,0.000000e+00,1.085000e+03,1.085000e+03,0.004574,0,1.000,0.945927,4.694409e+00,0.000000e+00
340890,AA23020RL74Y,2024-09-30,2021-09-30,2023-03-31,0.185,Stage 1,CONSUMER,0.054073,0.000000e+00,6.360000e+02,37,0.000000e+00,6.360000e+02,6.360000e+02,0.004574,0,1.000,0.945927,2.751746e+00,0.000000e+00
340891,AA23020VQ1DT,2024-09-30,2020-11-30,2024-09-30,0.190,Stage 1,CONSUMER,0.054073,2.150853e+06,3.230000e+02,47,0.000000e+00,3.230000e+02,3.230000e+02,0.004574,0,0.000,0.000000,0.000000e+00,0.000000e+00


In [ ]:
import xlsxwriter
with pd.ExcelWriter('Contractual Rundown.xlsx', engine = 'xlsxwriter') as writer:
    merged_df.to_excel(writer, sheet_name='Rundown',index= False)

In [ ]:
import xlsxwriter
with pd.ExcelWriter('PD Calcs v3.xlsx', engine = 'xlsxwriter') as writer:
    monthly_transition.to_excel(writer,sheet_name='Monthly transition',index = False)
    SummaryTransition.to_excel(writer,sheet_name='Transition', index = False)
    pd_per_month.to_excel(writer,sheet_name='Proportion Matrices', index = False)
    pd_normalized.to_excel(writer,sheet_name='Normalized', index = False)
    sorted_matrix.to_excel(writer,sheet_name='Marginal PD', index = False)

In [ ]:
from tkinter import Tk
from tkinter.filedialog import askopenfilename
from dateutil.relativedelta import relativedelta
import calendar
import pandas as pd
import numpy as np



class ECL_Modelling:
    def __init__(self):
        self.df = None
        

    def loading(self):
        Tk().withdraw()
        print("Please select the historical data to load") 
        file = askopenfilename()
        self.df = pd.read_excel(file, header=0)
        
        
    def generate_snapshot_dates(self,row):
        snapshot_dates = []
        current_date = row['Reporting_date']
        maturity_date = row['Adjusted_maturity_date']
        Stage = row['Stage']
        
        if Stage == 'Stage 1':
            for _ in range (13):
                snapshot_dates.append(current_date)
                current_date += relativedelta(months=1, day= 31)
                if current_date > maturity_date:#-relativedelta(months=3):
                       break

        elif Stage in ['Stage 2']:
            while current_date <= (maturity_date):#+relativedelta(months=3)
                snapshot_dates.append(current_date)
                current_date += relativedelta (months =1,day=31)
                if current_date > maturity_date:
                    break
        
            else:
                snapshot_dates = current_date

        return pd.Series(snapshot_dates)
    

    
    def calculate_period_since_origination(self, row):
            return ((row['Snapshot Dates'].year - row['First_payment'].year) * 12 +
                 (row['Snapshot Dates'].month - row['First_payment'].month))+1


    def calculate_monthly_installment(self, row):
        # Calculate the number of months between the first payment and maturity date
      # Calculate the period since origination
        n_months = ((row['Reporting_date'].year - row['First_payment'].year) * 12 + 
            (row['Reporting_date'].month - row['First_payment'].month))
        
        term_to_maturity = ((row['Adjusted_maturity_date'].year - row['First_payment'].year) * 12 + 
                    (row['Adjusted_maturity_date'].month - row['First_payment'].month))
        
        # Calculate the difference
        months_difference = term_to_maturity - n_months

        # Ensure n_months is not zero to avoid division by zero
        if months_difference == 0:
           return 0
        
        # Convert annual interest rate to monthly and calculate the installment
        monthly_rate = row['Effective_IR'] / 12
        if monthly_rate > 0:
                installment = (row['Outstanding_Amount'] * monthly_rate) / (1 - (1 + monthly_rate) ** -months_difference)
            
        else:
            installment = row['Outstanding_Amount'] / months_difference
        
        return installment
    
    #def adjusted_installment(self,row):
        #Frequency = self.df['Repayment_frequency'].iloc(0)
        #if Frequency == 'HYR':

    
    def calculate_balance(self, row, previous_balance, previous_account_no,monthly_installment):
        if row['TRANS_ID'] != previous_account_no:
            return row['Outstanding_Amount']
        else:
            return previous_balance * (1 + row['Effective_IR'] / 12) - monthly_installment
    
    def calculate_balance_after_missed_payment(df):
        balances_after_missed_payment = []
        for trans_id in df['TRANS_ID'].unique():
        #trans_df = df[.df['TRANS_ID'] == trans_id].reset_index(drop=True)
            for i in range(len(df)):
                offset_index = i - min (3,df.loc[i,''] - df.loc[0,'Period since origination'])
                balance = df.loc[offset_index, 'Balance after repayment'] * (1 + df['Effective_EIR']) ** min(3, df.loc[i, 'Period since origination'] - df.loc[0, 'Period since origination'])
                balances_after_missed_payment.append(balance)

class MergedDataframeHandler:
    def __init__(self, merged_df):
        self.merged_df = merged_df

    def merge_dataframes(self, final_df, sorted_matrix):
        """
        Merges final_df and sorted_matrix on 'Class', 'Period since origination', and 'Stage' in final_df,
        and 'SECTOR', 'Month', and 'Transition' in sorted_matrix. Adds the 'Marginal_PD' column from sorted_matrix
        to final_df and drops the now-redundant columns from sorted_matrix.

        Parameters:
        final_df (pd.DataFrame): The final dataframe containing loan data.
        sorted_matrix (pd.DataFrame): The dataframe containing sector, month, and transition-specific Marginal_PD values.

        Returns:
        pd.DataFrame: The merged dataframe with the 'Marginal_PD' column added.
        """
         # Merge the two dataframes on specified columns
        merged_df = final_df.merge(
            sorted_matrix[['SECTOR', 'Month', 'Marginal_PD', 'Transition']],  # Select only relevant columns from sorted_matrix
            left_on=['Class', 'Period since origination', 'Stage'],           # Columns in final_df to match
            right_on=['SECTOR', 'Month', 'Transition'],                       # Columns in sorted_matrix to match
            how='left')                                                 # Use left join to keep all rows from final_df
        
        # Drop the now-redundant columns from sorted_matrix
        merged_df.drop(columns=['SECTOR', 'Month', 'Transition'], inplace=True, errors='ignore')

        return merged_df

    def perform_new_calculations(self):
        # LGW and LGD Calculation
        self.merged_df['LGW'] = np.where(self.merged_df['Balance after missed payment']!=0,np.maximum(0,(1-(self.merged_df['Total DFSV']/self.merged_df['Balance after missed payment']))),0)
        self.merged_df['LGD'] = np.maximum(0,self.merged_df['LGW']*(1-self.merged_df['Cure rate']))
        #self.merged_df['Probability of not defaulting'] = 1- self.merged_df['Marginal_PD']
        self.merged_df['Credit Loss'] = self.merged_df['Balance after missed payment']*self.merged_df['Marginal_PD']*self.merged_df['LGD']
        self.merged_df['Discounted ECL'] = self.merged_df.apply(self.calculate_discounted_ecl, axis=1)

    def create_period_to_be_discounted(self):
    # Create a new column for period to be discounted
        self.merged_df['Period to be discounted'] = (
        self.merged_df.groupby('ACCOUNT NO').cumcount())



    def calculate_discounted_ecl(self, row):
        stage = row['Stage']
        if stage == 'Stage 3':
            return row['Credit Loss']
        if stage in ['Stage 1', 'Stage 2']:
            eir = row['EIR']
            period_to_discount = row['Period to be discounted']

            if period_to_discount > 0:
                return row['Credit Loss'] * (1 + eir / 12) ** - period_to_discount
        return 0
        

ecl_model = ECL_Modelling()
ecl_model.loading()
    
# Preallocate arrays
account_numbers = []
snapshot_dates = []
first_payments = []
maturity_dates = []
EIRs = []
stages = []
classes = []
cure_rates = []
dfsv =[]
balances = []
periods_since_origination = []
monthly_installments = []
adjusted_balances = []

# Iterate over rows in allocations_df

previous_account_no = None
previous_balance = 0


for index, row in ecl_model.df.iterrows():
    snapshots = ecl_model.generate_snapshot_dates(row)
    length = len(snapshots)
    
    # Extend the lists
    account_numbers.extend([row['TRANS_ID']] * length)
    snapshot_dates.extend(snapshots)
    first_payments.extend([row['First_payment']] * length)
    maturity_dates.extend([row['Adjusted_maturity_date']] * length)
    EIRs.extend([row['Effective_IR']] * length)
    stages.extend([row['Stage']] * length)
    classes.extend([row['Sector']] * length)
    cure_rates.extend([row['Cure_rate']] * length)
    dfsv.extend([row['Total_DFSV']]*length)
    balances.extend([row['Outstanding_Amount']] * length)

    monthly_installment = ecl_model.calculate_monthly_installment(row)


    for snapshot_date in snapshots:
        period = ecl_model.calculate_period_since_origination({
            'Snapshot Dates': snapshot_date,
            'First_payment': row['First_payment']
        })
        periods_since_origination.append (period)
        monthly_installments.append(ecl_model.calculate_monthly_installment(row))
        adjusted_balance = ecl_model.calculate_balance(row, previous_balance, previous_account_no,monthly_installment)
        adjusted_balances.append(adjusted_balance)
        previous_balance = adjusted_balance
        previous_account_no = row['TRANS_ID']

balances_after_missed_payment = ecl_model.calculate_balance_after_missed_payment(adjusted_balances, 
                                                                                EIRs, 
                                                                                account_numbers,
                                                                                stages)

# Create the DataFrame in one go
final_df = pd.DataFrame({
    'ACCOUNT NO': account_numbers,
    'Snapshot Dates': snapshot_dates,
    'First_payment': first_payments,
    'Maturity date': maturity_dates,
    'EIR': EIRs,
    'Stage': stages,
    'Class': classes,
    'Cure rate': cure_rates,
    'Total DFSV': dfsv,
    'Original Outstanding Balance': balances,
    'Period since origination':periods_since_origination,
    'Monthly installment': monthly_installments,
    'Balance after repayment':adjusted_balances,
    'Balance after missed payment': balances_after_missed_payment
})

# Merge with sorted_matrix
merged_handler = MergedDataframeHandler(None)
merged_df = merged_handler.merge_dataframes(final_df, sorted_matrix)

# Handle the merged dataframe
merged_handler.merged_df = merged_df
merged_handler.create_period_to_be_discounted()
merged_handler.perform_new_calculations()



Please select the historical data to load


TypeError: ECL_Modelling.calculate_balance_after_missed_payment() takes 1 positional argument but 5 were given

In [ ]:
print(f"ACCOUNT NO length: {len(account_numbers)}")
print(f"Snapshot Dates length: {len(snapshot_dates)}")
print(f"First_payment length: {len(first_payments)}")
print(f"Maturity date length: {len(maturity_dates)}")
print(f"EIR length: {len(EIRs)}")
print(f"Stage length: {len(stages)}")
print(f"Class length: {len(classes)}")
print(f"Cure rate length: {len(cure_rates)}")
print(f"Total DFSV length: {len(dfsv)}")
print(f"Original Outstanding Balance length: {len(balances)}")
print(f"Period since origination length: {len(periods_since_origination)}")
print(f"Monthly installment length: {len(monthly_installments)}")
print(f"Balance after repayment length: {len(adjusted_balances)}")
print(f"Balance after missed payment length: {len(balances_after_missed_payment)}")

ACCOUNT NO length: 340923
Snapshot Dates length: 340923
First_payment length: 340923
Maturity date length: 340923
EIR length: 340923
Stage length: 340923
Class length: 340923
Cure rate length: 340923
Total DFSV length: 340923
Original Outstanding Balance length: 340923
Period since origination length: 340923
Monthly installment length: 340923
Balance after repayment length: 340923
Balance after missed payment length: 267354


In [ ]:
from tkinter import Tk
from tkinter.filedialog import askopenfilename
from dateutil.relativedelta import relativedelta
import calendar
import pandas as pd
import numpy as np



class ECL_Modelling:
    def __init__(self):
        self.df = None
        

    def loading(self):
        Tk().withdraw()
        print("Please select the historical data to load") 
        file = askopenfilename()
        self.df = pd.read_excel(file, header=0)
        
        
    def generate_snapshot_dates(self,row):
        snapshot_dates = []
        current_date = row['Reporting_date']
        maturity_date = row['Adjusted_maturity_date']
        Stage = row['Stage']
        Frequency = self.df['Repayment_frequency'].iloc[0]
        if Frequency == 'QTR':
            increment = 3
            max_iterations = 5
        elif Frequency == 'HYR':
            increment = 6
            max_iterations = 3
        else:
            increment = 1
            max_iterations = 13

        
        if Stage == 'Stage 1':
            for _ in range (max_iterations):
                snapshot_dates.append(current_date)
                current_date += relativedelta(months=increment, day= 31)
                if current_date > maturity_date:#-relativedelta(months=3):
                       break

        elif Stage in ['Stage 2']:
            while current_date <= (maturity_date):#+relativedelta(months=3)
                snapshot_dates.append(current_date)
                current_date += relativedelta (months =increment,day=31)
        
            else:
                snapshot_dates = current_date

        return pd.Series(snapshot_dates)
    

    
    def calculate_period_since_origination(self, row):
        Frequency = self.df['Repayment_frequency'].iloc[0]
        if Frequency == 'QTR':
            return ((row['Snapshot Dates'].year - row['First_payment'].year) * 12 +
                 (row['Snapshot Dates'].month - row['First_payment'].month))+3
        
        elif Frequency == 'HYR':
            return ((row['Snapshot Dates'].year - row['First_payment'].year) * 12 +
                 (row['Snapshot Dates'].month - row['First_payment'].month))+6
        else:
            return ((row['Snapshot Dates'].year - row['First_payment'].year) * 12 +
                 (row['Snapshot Dates'].month - row['First_payment'].month))+1


    def calculate_monthly_installment(self, row):
        # Calculate the number of months between the first payment and maturity date
      # Calculate the period since origination
        n_months = ((row['Reporting_date'].year - row['First_payment'].year) * 12 + 
            (row['Reporting_date'].month - row['First_payment'].month))
        
        term_to_maturity = ((row['Adjusted_maturity_date'].year - row['First_payment'].year) * 12 + 
                    (row['Adjusted_maturity_date'].month - row['First_payment'].month))
        
        # Calculate the difference
        months_difference = term_to_maturity - n_months

        # Ensure n_months is not zero to avoid division by zero
        if months_difference == 0:
           return 0
        
        # Convert annual interest rate to monthly and calculate the installment
        monthly_rate = row['Effective_IR'] / 12
        Frequency = self.df['Repayment_frequency'].iloc[0]
        if monthly_rate > 0:
            if Frequency == 'QTR':
                installment = ((row['Outstanding_Amount'] * monthly_rate) / (1 - (1 + monthly_rate) ** -months_difference))*3
            elif Frequency == 'HYR':
                installment = ((row['Outstanding_Amount'] * monthly_rate) / (1 - (1 + monthly_rate) ** -months_difference))*6
            else:
                installment = (row['Outstanding_Amount'] * monthly_rate) / (1 - (1 + monthly_rate) ** -months_difference)
            
        else:
            installment = row['Outstanding_Amount'] / months_difference
        
        return installment
    
    def calculate_balance(self, row, previous_balance, previous_account_no,monthly_installment):
        Frequency = self.df['Repayment_frequency'].iloc[0]
        if row['TRANS_ID'] != previous_account_no:
            return row['Outstanding_Amount']
        else:
            if Frequency == 'QTR':
                return previous_balance * ((1 + row['Effective_IR'])**(3/12)) - monthly_installment
            elif Frequency == 'HYR':
                return previous_balance * ((1 + row['Effective_IR'])**(1/2)) - monthly_installment
            else:
                return previous_balance * (1 + row['Effective_IR'] / 12) - monthly_installment
    ''' 
    def calculate_balance_after_missed_payment(self, balances_after_repayment, eir, cust_ids, stages):
        balances_after_missed_payment = []
        
        for i in range(len(balances_after_repayment)):
            if stages[i] == 'Stage 1' and (i == 0 or cust_ids[i] != cust_ids[i-1]):
                balances_after_missed_payment.append(balances_after_repayment[i])
            elif stages[i] == 'Stage 1':
                if i % 5 == 1 or i % 5 == 2:
                    balances_after_missed_payment.append(balances_after_repayment[i-1] * (1 + eir[i] / 12 ))
                elif i % 5 == 3 or i % 5 == 4:
                    balances_after_missed_payment.append(balances_after_repayment[i-2] * (1 + eir[i] / 12 ) ** 2)
                else:
                    balances_after_missed_payment.append(balances_after_repayment[i])
            elif stages[i] in ['Stage 2A', 'Stage 2B'] and (i == 0 or cust_ids[i] != cust_ids[i-1]):
                balances_after_missed_payment.append(balances_after_repayment[i])
            elif stages[i] in ['Stage 2A', 'Stage 2B']:
                balances_after_missed_payment.append(balances_after_repayment[i-1] * (1 + eir[i] / 12 ))
            elif stages[i] == 'Default':
                balances_after_missed_payment.append(balances_after_repayment[i])
        
        return balances_after_missed_payment
'''

class MergedDataframeHandler:
    def __init__(self, merged_df):
        self.merged_df = merged_df

    def merge_dataframes(self, final_df, sorted_matrix):
        """
        Merges final_df and sorted_matrix on 'Class', 'Period since origination', and 'Stage' in final_df,
        and 'SECTOR', 'Month', and 'Transition' in sorted_matrix. Adds the 'Marginal_PD' column from sorted_matrix
        to final_df and drops the now-redundant columns from sorted_matrix.

        Parameters:
        final_df (pd.DataFrame): The final dataframe containing loan data.
        sorted_matrix (pd.DataFrame): The dataframe containing sector, month, and transition-specific Marginal_PD values.

        Returns:
        pd.DataFrame: The merged dataframe with the 'Marginal_PD' column added.
        """
         # Merge the two dataframes on specified columns
        merged_df = final_df.merge(
            sorted_matrix[['SECTOR', 'Month', 'Marginal_PD', 'Transition']],  # Select only relevant columns from sorted_matrix
            left_on=['Class', 'Period since origination', 'Stage'],           # Columns in final_df to match
            right_on=['SECTOR', 'Month', 'Transition'],                       # Columns in sorted_matrix to match
            how='left')                                                 # Use left join to keep all rows from final_df
        
        # Drop the now-redundant columns from sorted_matrix
        merged_df.drop(columns=['SECTOR', 'Month', 'Transition'], inplace=True, errors='ignore')

        return merged_df

    def perform_new_calculations(self):
        # LGW and LGD Calculation
        self.merged_df['LGW'] = np.where(self.merged_df['Balance after missed payment']!=0,np.maximum(0,(1-(self.merged_df['Total DFSV']/self.merged_df['Balance after missed payment']))),0)
        self.merged_df['LGD'] = np.maximum(0,self.merged_df['LGW']*(1-self.merged_df['Cure rate']))
        #self.merged_df['Probability of not defaulting'] = 1- self.merged_df['Marginal_PD']
        self.merged_df['Credit Loss'] = self.merged_df['Balance after missed payment']*self.merged_df['Marginal_PD']*self.merged_df['LGD']
        self.merged_df['Discounted ECL'] = self.merged_df.apply(self.calculate_discounted_ecl, axis=1)

    def create_period_to_be_discounted(self):
    # Create a new column for period to be discounted
        self.merged_df['Period to be discounted'] = (
        self.merged_df.groupby('ACCOUNT NO').cumcount())



    def calculate_discounted_ecl(self, row):
        stage = row['Stage']
        if stage == 'Stage 3':
            return row['Credit Loss']
        if stage in ['Stage 1', 'Stage 2']:
            eir = row['EIR']
            period_to_discount = row['Period to be discounted']

            if period_to_discount > 0:
                return row['Credit Loss'] * (1 + eir / 12) ** - period_to_discount
        return 0
        

ecl_model = ECL_Modelling()
ecl_model.loading()
    
# Preallocate arrays
account_numbers = []
snapshot_dates = []
first_payments = []
maturity_dates = []
EIRs = []
stages = []
classes = []
cure_rates = []
dfsv =[]
balances = []
periods_since_origination = []
monthly_installments = []
adjusted_balances = []

# Iterate over rows in allocations_df

previous_account_no = None
previous_balance = 0


for index, row in ecl_model.df.iterrows():
    snapshots = ecl_model.generate_snapshot_dates(row)
    length = len(snapshots)
    
    # Extend the lists
    account_numbers.extend([row['TRANS_ID']] * length)
    snapshot_dates.extend(snapshots)
    first_payments.extend([row['First_payment']] * length)
    maturity_dates.extend([row['Adjusted_maturity_date']] * length)
    EIRs.extend([row['Effective_IR']] * length)
    stages.extend([row['Stage']] * length)
    classes.extend([row['Sector']] * length)
    cure_rates.extend([row['Cure_rate']] * length)
    dfsv.extend([row['Total_DFSV']]*length)
    balances.extend([row['Outstanding_Amount']] * length)

    monthly_installment = ecl_model.calculate_monthly_installment(row)


    for snapshot_date in snapshots:
        period = ecl_model.calculate_period_since_origination({
            'Snapshot Dates': snapshot_date,
            'First_payment': row['First_payment']
        })
        periods_since_origination.append (period)
        monthly_installments.append(ecl_model.calculate_monthly_installment(row))
        adjusted_balance = ecl_model.calculate_balance(row, previous_balance, previous_account_no,monthly_installment)
        adjusted_balances.append(adjusted_balance)
        previous_balance = adjusted_balance
        previous_account_no = row['TRANS_ID']

balances_after_missed_payment = ecl_model.calculate_balance_after_missed_payment(adjusted_balances, 
                                                                                EIRs, 
                                                                                account_numbers,
                                                                                stages)

# Create the DataFrame in one go
final_df = pd.DataFrame({
    'ACCOUNT NO': account_numbers,
    'Snapshot Dates': snapshot_dates,
    'First_payment': first_payments,
    'Maturity date': maturity_dates,
    'EIR': EIRs,
    'Stage': stages,
    'Class': classes,
    'Cure rate': cure_rates,
    'Total DFSV': dfsv,
    'Original Outstanding Balance': balances,
    'Period since origination':periods_since_origination,
    'Monthly installment': monthly_installments,
    'Balance after repayment':adjusted_balances,
    'Balance after missed payment': balances_after_missed_payment
})

# Merge with sorted_matrix
merged_handler = MergedDataframeHandler(None)
merged_df = merged_handler.merge_dataframes(final_df, sorted_matrix)

# Handle the merged dataframe
merged_handler.merged_df = merged_df
merged_handler.create_period_to_be_discounted()
merged_handler.perform_new_calculations()



In [ ]:
from tkinter import Tk
from tkinter.filedialog import askopenfilename
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np

class ECL_Modelling:
    def __init__(self):
        self.df = None

    def loading(self):
        Tk().withdraw()
        print("Please select the historical data to load") 
        file = askopenfilename()
        self.df = pd.read_excel(file, header=0)

    def generate_snapshot_dates(self, row):
        snapshot_dates = []
        current_date = row['Reporting_date']
        maturity_date = row['Adjusted_maturity_date']
        stage = row['Stage']

        if stage == 'Stage 1':
            for _ in range(13):
                snapshot_dates.append(current_date)
                current_date += relativedelta(months=1, day=31)
                if current_date > maturity_date:
                    break
        elif stage in ['Stage 2']:
            while current_date <= maturity_date:
                snapshot_dates.append(current_date)
                current_date += relativedelta(months=1, day=31)
        
        return pd.Series(snapshot_dates)

    def calculate_period_since_origination(self, row):
        return ((row['Snapshot Dates'].year - row['First_payment'].year) * 12 +
                (row['Snapshot Dates'].month - row['First_payment'].month)) + 1

    def calculate_monthly_installment(self, row):
        n_months = ((row['Reporting_date'].year - row['First_payment'].year) * 12 + 
                     (row['Reporting_date'].month - row['First_payment'].month))

        term_to_maturity = ((row['Adjusted_maturity_date'].year - row['First_payment'].year) * 12 + 
                            (row['Adjusted_maturity_date'].month - row['First_payment'].month))
        
        months_difference = term_to_maturity - n_months

        if months_difference <= 0:
            return 0

        monthly_rate = row['Effective_IR'] / 12
        if monthly_rate > 0:
            installment = (row['Outstanding_Amount'] * monthly_rate) / (1 - (1 + monthly_rate) ** -months_difference)
        else:
            installment = row['Outstanding_Amount'] / months_difference
        
        return installment
    
    def calculate_balance(self, row, previous_balance, previous_account_no, monthly_installment):
        if row['TRANS_ID'] != previous_account_no:
            return row['Outstanding_Amount']
        else:
            return previous_balance * (1 + row['Effective_IR'] / 12) - monthly_installment
    
    def calculate_balance_after_missed_payment(self, adjusted_balances, EIRs, account_numbers, stages):
        balances_after_missed_payment = []
        for trans_id in np.unique(account_numbers):
            trans_df = self.df[self.df['TRANS_ID'] == trans_id]
            for i in range(len(trans_df)):
                offset_index = i - min(3, trans_df.loc[i, 'Period since origination'] - trans_df.loc[0, 'Period since origination'])
                if offset_index >= 0:
                    balance = trans_df.loc[offset_index, 'Balance after repayment'] * (1 + EIRs[i]) ** min(3, trans_df.loc[i, 'Period since origination'] - trans_df.loc[0, 'Period since origination'])
                else:
                    balance = trans_df.loc[i, 'Balance after repayment']
                balances_after_missed_payment.append(balance)
        return balances_after_missed_payment

class MergedDataframeHandler:
    def __init__(self, merged_df):
        self.merged_df = merged_df

    def merge_dataframes(self, final_df, sorted_matrix):
        merged_df = final_df.merge(
            sorted_matrix[['SECTOR', 'Month', 'Marginal_PD', 'Transition']],
            left_on=['Class', 'Period since origination', 'Stage'],
            right_on=['SECTOR', 'Month', 'Transition'],
            how='left'
        )
        merged_df.drop(columns=['SECTOR', 'Month', 'Transition'], inplace=True, errors='ignore')
        return merged_df

    def perform_new_calculations(self):
        self.merged_df['LGW'] = np.where(self.merged_df['Balance after missed payment'] != 0,
                                          np.maximum(0, (1 - (self.merged_df['Total DFSV'] / self.merged_df['Balance after missed payment']))), 0)
        self.merged_df['LGD'] = np.maximum(0, self.merged_df['LGW'] * (1 - self.merged_df['Cure rate']))
        self.merged_df['Credit Loss'] = self.merged_df['Balance after missed payment'] * self.merged_df['Marginal_PD'] * self.merged_df['LGD']
        self.merged_df['Discounted ECL'] = self.merged_df.apply(self.calculate_discounted_ecl, axis=1)

    def create_period_to_be_discounted(self):
        self.merged_df['Period to be discounted'] = self.merged_df.groupby('ACCOUNT NO').cumcount()

    def calculate_discounted_ecl(self, row):
        if row['Stage'] == 'Stage 3':
            return row['Credit Loss']
        if row['Stage'] in ['Stage 1', 'Stage 2']:
            eir = row['EIR']
            period_to_discount = row['Period to be discounted']
            if period_to_discount > 0:
                return row['Credit Loss'] * (1 + eir / 12) ** -period_to_discount
        return 0

# Main Execution
ecl_model = ECL_Modelling()
ecl_model.loading()

# Preallocate arrays
account_numbers, snapshot_dates, first_payments, maturity_dates = [], [], [], []
EIRs, stages, classes, cure_rates, dfsv, balances = [], [], [], [], [], []
periods_since_origination, monthly_installments, adjusted_balances = [], [], []

previous_account_no = None
previous_balance = 0

for index, row in ecl_model.df.iterrows():
    snapshots = ecl_model.generate_snapshot_dates(row)
    length = len(snapshots)

    account_numbers.extend([row['TRANS_ID']] * length)
    snapshot_dates.extend(snapshots)
    first_payments.extend([row['First_payment']] * length)
    maturity_dates.extend([row['Adjusted_maturity_date']] * length)
    EIRs.extend([row['Effective_IR']] * length)
    stages.extend([row['Stage']] * length)
    classes.extend([row['Sector']] * length)
    cure_rates.extend([row['Cure_rate']] * length)
    dfsv.extend([row['Total_DFSV']] * length)
    balances.extend([row['Outstanding_Amount']] * length)

    monthly_installment = ecl_model.calculate_monthly_installment(row)

    for snapshot_date in snapshots:
        period = ecl_model.calculate_period_since_origination({
            'Snapshot Dates': snapshot_date,
            'First_payment': row['First_payment']
        })
        periods_since_origination.append(period)
        monthly_installments.append(monthly_installment)
        adjusted_balance = ecl_model.calculate_balance(row, previous_balance, previous_account_no, monthly_installment)
        adjusted_balances.append(adjusted_balance)
        previous_balance = adjusted_balance
        previous_account_no = row['TRANS_ID']

# Create final DataFrame
final_df = pd.DataFrame({
    'ACCOUNT NO': account_numbers,
    'Snapshot Dates': snapshot_dates,
    'First_payment': first_payments,
    'Maturity date': maturity_dates,
    'EIR': EIRs,
    'Stage': stages,
    'Class': classes,
    'Cure rate': cure_rates,
    'Total DFSV': dfsv,
    'Original Outstanding Balance': balances,
    'Period since origination': periods_since_origination,
    'Monthly installment': monthly_installments,
    'Balance after repayment': adjusted_balances
})

# Merge with sorted_matrix
sorted_matrix = pd.DataFrame()  # Ensure sorted_matrix is defined
merged_handler = MergedDataframeHandler(None)
merged_df = merged_handler.merge_dataframes(final_df, sorted_matrix)

# Handle the merged dataframe
merged_handler.merged_df = merged_df
merged_handler.create_period_to_be_discounted()
merged_handler.perform_new_calculations()


Please select the historical data to load


KeyError: "None of [Index(['SECTOR', 'Month', 'Marginal_PD', 'Transition'], dtype='object')] are in the [columns]"